# Build Birdcall Mix

Pitch-shift `birdcall-chirp.wav` from the analyzed voice events, arrange harmonic chirps at the detected onset times, and blend the aligned consonant texture into `data/processed/mix.wav`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

from voice_to_birdcall.paths import PROCESSED_DATA_DIR

SR = 44_100
BIRD_GAIN = 0.6
CONSONANT_GAIN = 0.25
HARMONY_INTERVALS = [0, 7, 12]
HARMONY_GAINS = [1.0, 0.35, 0.18]
HARMONY_DELAYS = [0.0, 0.14, 0.28]

voice_path = PROCESSED_DATA_DIR / "voice.wav"
consonant_path = PROCESSED_DATA_DIR / "consonant.wav"
events_path = PROCESSED_DATA_DIR / "voice-events.json"
chirp_path = PROCESSED_DATA_DIR / "birdcall-chirp.wav"
mix_path = PROCESSED_DATA_DIR / "mix.wav"

# Load Intermediate Artifacts

Load the normalized voice and chirp, preserve the stored consonant amplitude, and restore event timing and pitch metadata from JSON.

In [2]:
def finite_audio(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if not np.all(np.isfinite(y)):
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y


def normalize_audio(y: np.ndarray) -> np.ndarray:
    y = finite_audio(y)
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    return y if peak < 1e-8 else y / peak


def load_normalized_audio(path: Path) -> np.ndarray:
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio file is empty: {path.name}")
    return normalize_audio(y)


def load_preserved_wav(path: Path) -> np.ndarray:
    y, sample_rate = sf.read(str(path), dtype="float32", always_2d=False)
    if sample_rate != SR:
        raise ValueError(f"Unexpected sample rate for {path.name}: {sample_rate}")
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    return finite_audio(y)


required_paths = [voice_path, consonant_path, events_path, chirp_path]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact not found: {path}")

voice = load_normalized_audio(voice_path)
consonant = load_preserved_wav(consonant_path)
base_chirp = load_normalized_audio(chirp_path)
analysis = json.loads(events_path.read_text(encoding="utf-8"))

if analysis["sample_rate"] != SR:
    raise ValueError(f"Unexpected event sample rate: {analysis['sample_rate']}")

events = analysis["events"]
median_pitch = float(analysis["median_pitch_hz"])
if not events:
    raise ValueError("No voice events found in voice-events.json.")

print(f"Loaded {len(events)} voice events at a median pitch of {median_pitch:.1f} Hz.")

Loaded 16 voice events at a median pitch of 359.5 Hz.


# Build Harmonic Chirps

Map relative voice pitch to the source chirp and layer the base pitch, a perfect fifth, and an octave with delays of 0, 140, and 280 ms.

In [3]:
def pitch_shift_chirp(
    chirp: np.ndarray,
    voice_pitch: float,
    base_pitch: float,
) -> np.ndarray:
    if voice_pitch <= 0 or base_pitch <= 0:
        return chirp
    semitones = float(np.clip(12 * np.log2(voice_pitch / base_pitch), -6, 8))
    return librosa.effects.pitch_shift(y=chirp, sr=SR, n_steps=semitones)


def make_harmony_chirp(
    chirp: np.ndarray,
    voice_pitch: float,
    base_pitch: float,
) -> np.ndarray:
    length = len(chirp) + int(max(HARMONY_DELAYS) * SR)
    harmony = np.zeros(length, dtype=np.float32)
    for interval, gain, delay in zip(
        HARMONY_INTERVALS,
        HARMONY_GAINS,
        HARMONY_DELAYS,
    ):
        shifted_pitch = voice_pitch * (2 ** (interval / 12))
        shifted = pitch_shift_chirp(chirp, shifted_pitch, base_pitch)
        start = int(delay * SR)
        harmony[start : start + len(shifted)] += shifted * gain
    return normalize_audio(harmony)

# Arrange and Save Mix

Place one harmony chirp at each onset, scale it by relative RMS, blend the aligned consonant texture, and save a floating-point WAV for lossless handoff to post-processing.

In [4]:
mix = np.zeros(int((len(voice) / SR + 2.5) * SR), dtype=np.float32)
max_amplitude = max(max(event["amp"] for event in events), 1e-8)

for event in events:
    chirp = make_harmony_chirp(base_chirp, event["pitch"], median_pitch)
    chirp *= 0.35 + 0.65 * (event["amp"] / max_amplitude)
    start = int(event["time"] * SR)
    end = start + len(chirp)
    if end > len(mix):
        continue

    if end <= len(consonant):
        voice_consonant = consonant[start:end]
    else:
        voice_consonant = np.zeros_like(chirp)

    event_sound = np.clip(
        chirp * BIRD_GAIN + voice_consonant * CONSONANT_GAIN,
        -1.0,
        1.0,
    )
    mix[start:end] += event_sound

sf.write(str(mix_path), finite_audio(mix), SR, subtype="FLOAT")

print(f"Saved {len(events)} arranged birdcall events to: {mix_path}")
display(Audio(normalize_audio(mix), rate=SR))

c:\Users\morph\GitHub\voice-to-birdcall\.venv\Lib\site-packages\librosa\effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
c:\Users\morph\GitHub\voice-to-birdcall\.venv\Lib\site-packages\librosa\effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
c:\Users\morph\GitHub\voice-to-birdcall\.venv\Lib\site-packages\numba\np\ufunc\dufunc.py:303: RuntimeWarning: invalid value encountered in cast
  return super().__call__(*args, **kws)


Saved 16 arranged birdcall events to: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\mix.wav
